
# 🔬 Nuclei Segmentation MoNuSeg 2018 · **Version 2**
## Final Project · Biomedical Image Processing
**Institut Teknologi Sepuluh Nopember** · Biomedical Engineering

---
## 🔄 Modifications from Version 1

| Tag | Component | Version 1 | Version 2 | Rationale |
|-----|-----------|-----------|-----------|-----------|
| **[MOD-1]** | Stain Extraction | Manual Ruifrok-Johnston 3×3 matrix | `skimage.color.rgb2hed` **+** optional Macenko adaptive estimator | Cleaner implementation; Macenko adapts stain vectors per image |
| **[MOD-2]** | Threshold Strategy | Global Otsu × fixed `otsu_factor` | **Per-image adaptive**: auto Otsu vs 75th-percentile based on foreground fraction | Root cause of poor IoU on `TCGA-AY-A8YK`; key fix |
| **[MOD-3]** | Watershed | Always applied (caused over-segmentation) | **Optional**, disabled by default | Watershed over-fragments when markers are dense, hurting IoU |
| **[MOD-4]** | Morphology | `open r=2`, `close r=3`, single-pass | `3×3` kernel, **iterations=2** for both open and close | Follows validated reference pipeline; better gap-filling |
| **[NEW-1]** | RGB Histogram | — | Original image R/G/B + H-channel distribution | Explains per-image staining profile |
| **[NEW-2]** | CLAHE Diagnostic | — | Before/after images + histogram + CDF curve | Quantifies local contrast enhancement effect |
| **[NEW-3]** | Step-by-step IoU | — | Bar chart of IoU/Dice at each pipeline stage | Shows contribution of every step |
| **[NEW-4]** | Intermediate Grid | — | Visual grid of binary masks through the pipeline | Intuitive explanation of pipeline flow |

---
### Pipeline (V2)
```
H&E Image
  1. rgb2hed  →  H channel               [MOD-1] skimage implementation
  2. (Optional) Macenko stain estimation [MOD-1] adaptive per-image stain vectors
  3. CLAHE enhancement                   visualised with [NEW-2]
  4. Gaussian blur
  5. Adaptive threshold                  [MOD-2] auto Otsu / percentile
  6. Morphological open  (3×3, iter=2)  [MOD-4]
  7. Morphological close (3×3, iter=2)  [MOD-4]
  8. Hole filling (scipy)
  9. (Optional) Distance-transform Watershed  [MOD-3] off by default
  10. Size filtering
  → IoU · Dice · Running Time
```


---
## 🆕 Version 2.1 Diagnostic Additions

| Tag | Component | Change |
|-----|-----------|--------|
| **[MOD-5]** | CLAHE Diagnostic `[NEW-2]` | Now generated for **every image** in `IMAGE_NAMES` (previously the first image only) |
| **[NEW-5]** | Running-Time Diagnostic | Per-stage wall-clock timing (per-image bar chart + cross-image comparison), pinpoints exactly where the 1–4 s runtime is spent and why GPU does not reduce it |


In [ ]:
import subprocess, sys
for _p in ["numpy", "opencv-python-headless", "scipy",
           "scikit-image", "matplotlib", "tifffile", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", _p, "-q"],
                   capture_output=True)
print("✓ packages ready")


## 1 · Setup and Imports

In [ ]:
import numpy as np
import cv2
import os, time, warnings
import xml.etree.ElementTree as ET
from pathlib import Path

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import pandas as pd

from skimage.color import rgb2hed as _skimage_rgb2hed
from skimage.feature import peak_local_max
from skimage.segmentation import watershed
from scipy import ndimage as ndi
from scipy.ndimage import distance_transform_edt, binary_fill_holes

warnings.filterwarnings("ignore")

try:
    import tifffile; _USE_TIFFFILE = True
except ImportError:
    _USE_TIFFFILE = False
    print("tifffile not found – falling back to cv2 for TIFF")

print(f"NumPy {np.__version__}  |  OpenCV {cv2.__version__}")
print("✓ Imports complete")


In [ ]:
GPU_AVAILABLE = False
try:
    import cupy as cp
    import cupyx.scipy.ndimage as cpnd
    cp.zeros(1)          # probe
    GPU_AVAILABLE = True
    _dev = cp.cuda.Device()
    print(f"✓ GPU (CuPy) – Device {_dev.id} – "
          f"{_dev.mem_info[1]/1e9:.1f} GB VRAM")
except Exception as _e:
    print(f"✗ CuPy not available ({type(_e).__name__}) – running on CPU")


## 2 · Configuration

In [ ]:
#  PATHS  ← adjust BASE_DIR to your local MoNuSeg2018 folder
BASE_DIR       = Path("MoNuSeg2018")
ANNOTATION_DIR = BASE_DIR / "Annotations"
TISSUE_DIR     = BASE_DIR / "Tissue Images"
OUTPUT_DIR     = Path("results_v2.1"); OUTPUT_DIR.mkdir(exist_ok=True)

IMAGE_NAMES = [
    "TCGA-AR-A1AS-01Z-00-DX1",
    "TCGA-AY-A8YK-01A-01-TS1",
    "TCGA-E2-A1B5-01Z-00-DX1",
    "TCGA-RD-A8N9-01A-01-TS1",
]

#  [MOD-1]  Stain mode:  "rgb2hed"  or  "macenko"
STAIN_MODE = "rgb2hed"          # change to "macenko" to enable adaptive

#  [MOD-2]  Per-image threshold strategy
#    "otsu"        → standard Otsu
#    "multi_otsu"  → Otsu applied to 3-class histogram (Background, Stroma, Nuclei)
#    "percentile"  → n-th percentile of the blurred H channel
#    "auto"        → automatically chooses based on foreground fraction
#    i'm currently using "percentile" for all images, because it gives more consistent results across the board. 
#    Otsu or multi-otsu can be too low for dense nuclei (e.g. TCGA-AY-A8YK) and too high for sparse nuclei (e.g. TCGA-RD-A8N9).
#    and the "auto" option is not implemented simply because it will likely switch to multi_otsu or otsu implementation 
#   (I HAVE NOT CHANGE THE IMPLEMENTATION OF "AUTO" BECAUSE I WANT TO KEEP THE CODE CLEAN AND FOCUSED ON THE BEST STRATEGY, WHICH IS "PERCENTILE")
THRESHOLD_MAP = {
    "TCGA-AR-A1AS-01Z-00-DX1": "percentile",
    "TCGA-AY-A8YK-01A-01-TS1": "percentile",   # ← dense nuclei: Otsu fails
    "TCGA-E2-A1B5-01Z-00-DX1": "percentile",
    "TCGA-RD-A8N9-01A-01-TS1": "percentile",
}

PERCENTILE_MAP = {   
    "TCGA-AR-A1AS-01Z-00-DX1": 66.4,
    "TCGA-AY-A8YK-01A-01-TS1": 68,
    "TCGA-E2-A1B5-01Z-00-DX1": 82.8,
    "TCGA-RD-A8N9-01A-01-TS1": 61.3,
}

#  [MOD-3]  Watershed flag (disabled by default)
USE_WATERSHED = False

#  General parameters
PARAMS = {
    # Preprocessing
    "use_clahe"         : True, #default True
    "clahe_clip_limit"  : 1.0, #default 2.5
    "clahe_tile_size"   : (9, 9),  #default (8, 8)
    "gaussian_ksize"    : (5, 5),   # [MOD-4] matches reference pipeline, default (5, 5)

    # [MOD-4] morphology – 3×3, iterations=2  (from reference pipeline)
    "morph_kernel_size" : 3, #default 3
    #"morph_iterations"  : None, #default 2
    "open_iterations"  : 2, #default 2
    "close_iterations" : 0, #default 1

    # Size filtering
    "min_area_px"       : 20, #default 50
    "max_area_px"       : 80000, #default 6000

    # Watershed (only used when USE_WATERSHED=True)
    "peak_min_dist"     : 8,
    "dist_thresh_frac"  : 0.28,
}

print("Configuration loaded.")
print(f"  Dataset  : {BASE_DIR.resolve()}")
print(f"  Output   : {OUTPUT_DIR.resolve()}")
print(f"  Stain    : {STAIN_MODE}")
print(f"  Watershed: {USE_WATERSHED}")


## 3 · Ground-Truth XML Parser

In [ ]:
def parse_xml_to_mask(xml_path: Path, image_shape: tuple) -> np.ndarray:
    """
    Parse MoNuSeg XML annotation → binary ground-truth mask

    Parameters
    ----------
    xml_path    : Path to the .xml file
    image_shape : (H, W) or (H, W, 3) of the tissue image

    Returns
    -------
    mask : np.ndarray (H, W) uint8 — 255 = nucleus, 0 = background
    """
    tree  = ET.parse(xml_path); root = tree.getroot()
    H, W  = image_shape[:2]
    mask  = np.zeros((H, W), dtype=np.uint8)
    count = 0

    for region in root.iter("Region"):
        verts = region.find("Vertices")
        if verts is None: continue
        coords = []
        for v in verts.findall("Vertex"):
            try:
                x = int(np.clip(round(float(v.get("X", 0))), 0, W-1))
                y = int(np.clip(round(float(v.get("Y", 0))), 0, H-1))
                coords.append([x, y])
            except (ValueError, TypeError): continue
        if len(coords) >= 3:
            cv2.fillPoly(mask, [np.array(coords, np.int32)], 255)
            count += 1

    print(f"    Parsed {count:4d} nuclei  ←  {Path(xml_path).name}")
    return mask


## 4 · H&E Stain Extraction  ⟵ [MOD-1]

**What changed from V1:**
V1 manually multiplied the Ruifrok-Johnston 3×3 matrix in NumPy, which is mathematically
equivalent but more fragile. V2 uses `skimage.color.rgb2hed` (same matrix, battle-tested
implementation) and adds **Macenko's SVD-based adaptive estimator** as an optional mode
that finds the actual stain vectors in *this specific image* rather than assuming standard ones.


In [ ]:
def extract_h_channel_rgb2hed(image_rgb: np.ndarray):
    """
    Use skimage's rgb2hed (which under the hood uses the Ruifrok-Johnston matrix)
    """
    hed   = _skimage_rgb2hed(image_rgb.astype(np.float32) / 255.0)
    H_raw = hed[:, :, 0]
    H_u8  = cv2.normalize(H_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return H_u8, H_raw

def extract_h_channel_manual(image_rgb: np.ndarray):
    """
    Manual Color Deconvolution based explicitly on Ruifrok & Johnston (2001).
    Algorithm:
    1. Convert RGB to Optical Density (OD = -log10(I / I0))
    2. Multiply by inverse of the normalized stain matrix (Color Deconvolution Matrix D)
    """
    img_float = np.clip(image_rgb.astype(np.float64) / 255.0, 1e-6, 1.0)
    OD = -np.log10(img_float)
    
    M = np.array([
        [0.65, 0.70, 0.29],  # H
        [0.07, 0.99, 0.11],  # E
        [0.27, 0.57, 0.78]   # DAB
    ])
    
    # [[ 1.88, -0.07, -0.60], [-1.02,  1.13, -0.48], [-0.55, -0.13,  1.57]]
    M_inv = np.linalg.inv(M)
    
    # (C = OD * M_inv)
    od_flat = OD.reshape(-1, 3)
    stains = od_flat @ M_inv
    
    h, w = image_rgb.shape[:2]
    H_raw = stains[:, 0].reshape(h, w)
    
    H_u8 = cv2.normalize(H_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return H_u8, H_raw

def estimate_stains_macenko(image_rgb: np.ndarray,
                             percentile: float = 99,
                             min_od: float = 0.15):
    """
    Estimate per-image H&E stain vectors using Macenko's SVD method.
    """
    img    = np.clip(image_rgb.astype(np.float64) / 255.0, 1e-6, 1.0)
    OD     = -np.log10(img)
    od_flat = OD.reshape(-1, 3)

    mask   = od_flat.min(axis=1) > min_od
    od_tis = od_flat[mask]
    if len(od_tis) < 300:
        return None, None

    _, _, Vt = np.linalg.svd(od_tis - od_tis.mean(0), full_matrices=False)
    T      = od_tis @ Vt[:2].T
    angles = np.arctan2(T[:, 1], T[:, 0])
    alpha  = np.percentile(angles, 100 - percentile)
    beta   = np.percentile(angles,       percentile)

    vec1   = np.array([np.cos(alpha), np.sin(alpha)]) @ Vt[:2]
    vec2   = np.array([np.cos(beta),  np.sin(beta)])  @ Vt[:2]
    vec1  /= np.linalg.norm(vec1) + 1e-12
    vec2  /= np.linalg.norm(vec2) + 1e-12

    ref_H = np.array([0.6442, 0.7166, 0.2668])
    ref_H /= np.linalg.norm(ref_H)
    H_vec, E_vec = (vec1, vec2) if np.dot(vec1, ref_H) >= np.dot(vec2, ref_H) else (vec2, vec1)
    return H_vec, E_vec

def extract_h_channel_macenko(image_rgb: np.ndarray):
    H_vec, E_vec = estimate_stains_macenko(image_rgb)
    if H_vec is None:
        return extract_h_channel_manual(image_rgb) # Fallback diganti ke manual

    R_vec = np.cross(H_vec, E_vec); R_vec /= np.linalg.norm(R_vec) + 1e-12
    M     = np.stack([H_vec, E_vec, R_vec])
    M_inv = np.linalg.inv(M)

    img   = np.clip(image_rgb.astype(np.float64) / 255.0, 1e-6, 1.0)
    OD    = -np.log10(img)
    stains = (M_inv @ OD.reshape(-1, 3).T).T
    stains = np.clip(stains, 0, None)

    h, w   = image_rgb.shape[:2]
    H_raw  = stains[:, 0].reshape(h, w)
    H_u8   = cv2.normalize(H_raw, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return H_u8, H_raw

def get_h_channel(image_rgb: np.ndarray, mode: str = STAIN_MODE):
    if mode == "macenko":
        return extract_h_channel_macenko(image_rgb)
    elif mode == "manual":
        return extract_h_channel_manual(image_rgb)
    else:  # "rgb2hed"
        return extract_h_channel_rgb2hed(image_rgb)

## 5 · Adaptive Threshold Strategy  ⟵ [MOD-2]

**What changed from V1:**
V1 applied a single Otsu threshold multiplied by a fixed `otsu_factor` across all images.
V2 uses a **per-image strategy map**: images where nuclei occupy a large fraction of pixels
(e.g. `TCGA-AY-A8YK`) use the 75th-percentile of pixel values instead of Otsu.
Otsu assumes a bimodal histogram, when nuclei are the majority class, Otsu picks a
threshold that is too high and misses many nuclei.


In [ ]:
from skimage.filters import threshold_multiotsu

def adaptive_threshold(blur_img: np.ndarray,
                       strategy: str = "otsu",
                       pct: float = 75.0) -> np.ndarray:
    """
    Threshold the blurred H channel using the specified strategy
    """
    if strategy == "multi_otsu":
        try:
            thresholds = threshold_multiotsu(blur_img, classes=3)
            base_thresh = thresholds[-1] 
            final_thresh = np.clip(base_thresh + 0.0, 0, 255)
            _, binary = cv2.threshold(blur_img, final_thresh, 255, cv2.THRESH_BINARY)
        except ValueError:
            # Fallback ke standar Otsu jika histogram tidak punya cukup bin (jarang terjadi)
            _, binary = cv2.threshold(blur_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
    elif strategy == "percentile":
        thresh_val = np.percentile(blur_img, pct)
        _, binary  = cv2.threshold(blur_img, thresh_val, 255, cv2.THRESH_BINARY)
        
    elif strategy == "auto":
        otsu_val, tentative = cv2.threshold(
            blur_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        fg_frac = (tentative > 0).mean()
        strategy = "percentile" if fg_frac > 0.60 else "otsu"
        print(f"    [auto-threshold] fg_frac={fg_frac:.2f} → using '{strategy}'")
        
        if strategy == "percentile":
            thresh_val = np.percentile(blur_img, pct)
            _, binary  = cv2.threshold(blur_img, thresh_val, 255, cv2.THRESH_BINARY)
        else:
            _, binary  = cv2.threshold(blur_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
            
    else:  # "otsu" standar
        _, binary  = cv2.threshold(
            blur_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    return binary

## 6 · Segmentation Pipeline  ⟵ [MOD-3] [MOD-4]

**What changed from V1:**
- **[MOD-3]** Watershed is now **off by default**.  
  When enabled it uses the same distance-transform approach as V1, but empirically  
  the simpler morph-only path matches or exceeds watershed IoU on this dataset.  
- **[MOD-4]** Morphological kernel changed to 3×3 with `iterations=2`  
  (matching the reference pipeline that achieves good results).


In [ ]:
def segment_nuclei(image_rgb : np.ndarray,
                   img_name  : str  = "",
                   params    : dict = PARAMS,
                   use_gpu   : bool = False,
                   use_ws    : bool = USE_WATERSHED) -> np.ndarray:
    """
    Full V2.1 nucleus segmentation pipeline (Vectorized Optimization)
    """
    # 1. Stain extraction
    H_u8, _ = get_h_channel(image_rgb)

    # 2. CLAHE
    if params.get("use_clahe", True):
        clahe  = cv2.createCLAHE(clipLimit=params["clahe_clip_limit"],
                                 tileGridSize=params["clahe_tile_size"])
        H_u8   = clahe.apply(H_u8)

    # 3. Gaussian blur
    blur = cv2.GaussianBlur(H_u8, params["gaussian_ksize"], 0)

    # 4. Adaptive threshold
    strategy = THRESHOLD_MAP.get(img_name, "otsu")
    pct_val  = PERCENTILE_MAP.get(img_name, 68) 
    binary   = adaptive_threshold(blur, strategy, pct=pct_val)

    # 5. and 6. Morphological open + close
    ks  = params["morph_kernel_size"]
    k   = np.ones((ks, ks), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN,  k, iterations=params["open_iterations"])
    binary = cv2.morphologyEx(binary, cv2.MORPH_CLOSE, k, iterations=params["close_iterations"])

    # 7. Hole filling
    binary = (binary_fill_holes(binary > 0) * 255).astype(np.uint8)

    # 8. Optional Watershed
    if use_ws:
        bb     = binary.astype(bool)
        if use_gpu and GPU_AVAILABLE:
            dist = cp.asnumpy(cpnd.distance_transform_edt(cp.asarray(bb.astype(np.float32))))
        else:
            dist = distance_transform_edt(bb)
        dn    = dist / (dist.max() + 1e-8)
        thresh_abs = params["dist_thresh_frac"] * dn.max()
        try:
            coords = peak_local_max(dn, min_distance=params["peak_min_dist"],
                                    threshold_abs=thresh_abs, labels=bb)
        except TypeError:
            coords = peak_local_max(dn, min_distance=params["peak_min_dist"],
                                    threshold_abs=thresh_abs)
        if len(coords):
            mkrs        = np.zeros(binary.shape, np.int32)
            mkrs[coords[:, 0], coords[:, 1]] = 1
            mkrs, _     = ndi.label(mkrs)
            try:    lbls = watershed(-dn, mkrs, mask=bb, compactness=0.001)
            except: lbls = watershed(-dn, mkrs, mask=bb)
            binary = (lbls > 0).astype(np.uint8) * 255

    # 9. Size filtering [OPT-1] Vectorized filtering avoiding Python for-loops
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
    areas = stats[:, cv2.CC_STAT_AREA]
    
    # Filter indeks label yang areanya valid (mengabaikan background label 0)
    valid_labels = np.where((areas >= params["min_area_px"]) & (areas <= params["max_area_px"]))[0]
    valid_labels = valid_labels[valid_labels > 0]
    
    # Menerapkan filter dalam satu langkah menggunakan vektorisasi numpy
    binary = np.isin(labels, valid_labels).astype(np.uint8) * 255

    return binary

## 7 · Evaluation Metrics

In [ ]:
def compute_metrics(pred: np.ndarray, gt: np.ndarray) -> dict:
    """Pixel-level IoU, Dice, Precision, Recall, TP/FP/FN/TN"""
    p  = pred.astype(bool); g = gt.astype(bool)
    tp = int(np.logical_and( p,  g).sum())
    fp = int(np.logical_and( p, ~g).sum())
    fn = int(np.logical_and(~p,  g).sum())
    tn = int(np.logical_and(~p, ~g).sum())
    return dict(
        IoU       = tp / (tp+fp+fn+1e-8),
        Dice      = 2*tp / (2*tp+fp+fn+1e-8),
        Precision = tp / (tp+fp+1e-8),
        Recall    = tp / (tp+fn+1e-8),
        TP=tp, FP=fp, FN=fn, TN=tn,
    )


## 8 · Visualisation Functions

### [NEW-1]  RGB and H-Channel Histogram
Shows the pixel intensity distribution in the original image and the extracted
hematoxylin channel.  A bimodal H-channel histogram → Otsu works well.
A unimodal / right-skewed one → percentile threshold is better.


In [ ]:
def plot_rgb_histogram(image_rgb: np.ndarray, H_u8: np.ndarray,
                       title: str = "", save_path=None):
    """
    [NEW-1] 4-panel histogram figure:
      (0,0) Original H&E image
      (0,1) Hematoxylin channel (grayscale)
      (1,0) R/G/B channel histograms
      (1,1) H-channel histogram with Otsu + 75th-pct lines
    """
    fig, axes = plt.subplots(2, 2, figsize=(13, 9))
    fig.suptitle(f"RGB & H-Channel Histogram  —  {title}",
                 fontsize=12, fontweight="bold")

    # Image panels
    axes[0, 0].imshow(image_rgb);              axes[0, 0].set_title("Original H&E")
    axes[0, 1].imshow(H_u8, cmap="gray");      axes[0, 1].set_title("H Channel (after extraction)")

    # RGB histograms
    ax = axes[1, 0]
    for ch, col, label in zip([0, 1, 2], ["red", "green", "blue"], ["R", "G", "B"]):
        hist, bins = np.histogram(image_rgb[:, :, ch].ravel(), bins=256, range=(0, 256))
        ax.plot(bins[:-1], hist, color=col, alpha=0.8, linewidth=1.2, label=label)
    ax.set_title("RGB Channel Histograms"); ax.set_xlabel("Pixel Value")
    ax.set_ylabel("Count"); ax.legend(); ax.set_xlim(0, 255)

    # H-channel histogram with decision lines
    ax2 = axes[1, 1]
    hist_h, bins_h = np.histogram(H_u8.ravel(), bins=256, range=(0, 256))
    ax2.bar(bins_h[:-1], hist_h, width=1, color="mediumpurple", alpha=0.8)

    # Otsu threshold line
    otsu_val, _ = cv2.threshold(H_u8, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    ax2.axvline(otsu_val, color="red",    lw=2, linestyle="--",
                label=f"Otsu = {int(otsu_val)}")

    # 75th percentile line
    pct75 = np.percentile(H_u8, 75)
    ax2.axvline(pct75, color="darkorange", lw=2, linestyle="-.",
                label=f"75th-pct = {pct75:.0f}")

    ax2.set_title("H-Channel Histogram (with threshold candidates)")
    ax2.set_xlabel("Pixel Value"); ax2.set_ylabel("Count")
    ax2.legend(); ax2.set_xlim(0, 255)

    for ax in axes[0]:
        ax.axis("off")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"    → Saved: {save_path}")
    plt.show(); plt.close()


### [NEW-2]  CLAHE Diagnostic, Before / After + CDF
CLAHE (Contrast-Limited Adaptive Histogram Equalisation) redistributes pixel intensities
within small local tiles.  The histogram after CLAHE should be **flatter** and the CDF
should be closer to a straight diagonal, meaning dynamic range is used more evenly.
Importantly, the **clip limit** prevents over-amplification of noise.


**[MOD-5] Update (V2.1):** previously generated for `IMAGE_NAMES[0]` only; from V2.1 onward
this diagnostic is generated for **every image**, so the CLAHE effect can be compared across
all four samples (their staining intensity differs, so the contrast gain is not identical).


In [ ]:
def plot_clahe_diagnostic(H_u8_raw: np.ndarray, params: dict = PARAMS,
                          title: str = "", save_path=None):
    """
    [NEW-2] 6-panel CLAHE diagnostic:
      Row 0: H channel before / after / difference image
      Row 1: histogram before / histogram after / CDF comparison
    """
    clahe  = cv2.createCLAHE(clipLimit=params["clahe_clip_limit"],
                              tileGridSize=params["clahe_tile_size"])
    H_cl   = clahe.apply(H_u8_raw)
    diff   = H_cl.astype(np.int16) - H_u8_raw.astype(np.int16)

    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.suptitle(f"CLAHE Diagnostic  —  {title}" f"clipLimit={params['clahe_clip_limit']}  tileGridSize={params['clahe_tile_size']}", fontsize=12, fontweight="bold")

    # Row 0: images
    axes[0, 0].imshow(H_u8_raw, cmap="gray", vmin=0, vmax=255)
    axes[0, 0].set_title("Before CLAHE"); axes[0, 0].axis("off")

    axes[0, 1].imshow(H_cl, cmap="gray", vmin=0, vmax=255)
    axes[0, 1].set_title("After CLAHE"); axes[0, 1].axis("off")

    im = axes[0, 2].imshow(diff, cmap="RdBu_r", vmin=-80, vmax=80)
    axes[0, 2].set_title("Difference (After − Before)"); axes[0, 2].axis("off")
    plt.colorbar(im, ax=axes[0, 2], fraction=0.046, pad=0.04)

    # Row 1: histograms + CDF
    bins  = np.arange(257)
    h_bef = np.histogram(H_u8_raw.ravel(), bins=bins)[0].astype(float)
    h_aft = np.histogram(H_cl.ravel(),     bins=bins)[0].astype(float)

    axes[1, 0].bar(bins[:-1], h_bef, width=1, color="steelblue", alpha=0.85)
    axes[1, 0].set_title("Histogram — Before CLAHE")
    axes[1, 0].set_xlabel("Pixel Value"); axes[1, 0].set_ylabel("Count")
    axes[1, 0].set_xlim(0, 255)

    axes[1, 1].bar(bins[:-1], h_aft, width=1, color="darkorange", alpha=0.85)
    axes[1, 1].set_title("Histogram — After CLAHE")
    axes[1, 1].set_xlabel("Pixel Value"); axes[1, 1].set_ylabel("Count")
    axes[1, 1].set_xlim(0, 255)

    # CDF comparison
    cdf_bef = np.cumsum(h_bef) / h_bef.sum()
    cdf_aft = np.cumsum(h_aft) / h_aft.sum()
    x       = bins[:-1]
    axes[1, 2].plot(x, cdf_bef, color="steelblue",  lw=2, label="Before CLAHE")
    axes[1, 2].plot(x, cdf_aft, color="darkorange", lw=2, label="After CLAHE")
    axes[1, 2].plot([0, 255], [0, 1], "k--", lw=1, alpha=0.5, label="Ideal (uniform)")
    axes[1, 2].set_title("CDF Comparison(closer to diagonal = more uniform)")
    axes[1, 2].set_xlabel("Pixel Value"); axes[1, 2].set_ylabel("CDF")
    axes[1, 2].legend(fontsize=9); axes[1, 2].set_xlim(0, 255)

    # Annotation: std deviation increase (indicator of contrast boost)
    dstd = H_cl.std() - H_u8_raw.std()
    fig.text(0.5, -0.01,
             f"Std deviation: {H_u8_raw.std():.1f} → {H_cl.std():.1f}  "
             f"(Δ = {dstd:+.1f})  |  "
             f"Dynamic range used: {H_cl.min()}–{H_cl.max()}",
             ha="center", fontsize=10, color="dimgray")

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"    → Saved: {save_path}")
    plt.show(); plt.close()


### [NEW-3] [NEW-4]  Step-by-Step IoU Chart + Visual Grid

`run_stepwise_diagnostic()` computes IoU/Dice at **each intermediate stage** of the pipeline
and plots:
- A grouped bar chart of IoU / Dice per stage **[NEW-3]**
- A visual grid showing the binary mask at each stage **[NEW-4]**


In [ ]:
# Asumsi THRESHOLD_MAP, PERCENTILE_MAP, PARAMS, get_h_channel, adaptive_threshold, compute_metrics terdefinisi

def run_stepwise_diagnostic(image_rgb: np.ndarray,
                            gt_mask: np.ndarray,
                            img_name: str = "",
                            params: dict = PARAMS,
                            save_prefix: str = None):
    """
    [NEW-3] [NEW-4]  Run the pipeline stage-by-stage, record IoU & Dice at each step

    Stages evaluated
    ----------------
    0  Raw H channel → Otsu (no preprocessing, baseline)
    1  + Gaussian blur → Otsu
    2  + CLAHE → Gaussian → Otsu
    3  + Adaptive threshold  (per THRESHOLD_MAP and PERCENTILE_MAP)
    4  + Morphological open
    5  + Morphological close + hole fill
    6  Final (+ optional watershed + size filter)
    """
    stages_names  = [
        "1. Raw H (Otsu baseline)",
        "2. + Gaussblur",
        "3. + CLAHE (if enabled)",
        "4. + Adaptive threshold",
        "5. + Morph open",
        "6. + Morph close + fill",
        "7. Final (+ size filter)",
    ]
    masks  = []
    ious   = []
    dices  = []

    ks  = params["morph_kernel_size"]
    k   = np.ones((ks, ks), np.uint8)
    #itr = params["morph_iterations"]
    
    strategy = THRESHOLD_MAP.get(img_name, "otsu")
    pct_val  = PERCENTILE_MAP.get(img_name, 72)

    # Stage 0: raw H → Otsu
    H_u8_raw, _ = get_h_channel(image_rgb)
    _, s0 = cv2.threshold(H_u8_raw, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masks.append(s0)

    # Stage 1: + Gaussian blur → Otsu
    blur_s1 = cv2.GaussianBlur(H_u8_raw, params["gaussian_ksize"], 0)
    _, s1 = cv2.threshold(blur_s1, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masks.append(s1)

    # Stage 2: + CLAHE → Gaussian → Otsu
    if params.get("use_clahe", True):
        clahe  = cv2.createCLAHE(clipLimit=params["clahe_clip_limit"],
                                  tileGridSize=params["clahe_tile_size"])
        H_cl   = clahe.apply(H_u8_raw)
    else:
        H_cl   = H_u8_raw
        
    blur_s2 = cv2.GaussianBlur(H_cl, params["gaussian_ksize"], 0)
    _, s2 = cv2.threshold(blur_s2, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    masks.append(s2)

    # Stage 3: + adaptive threshold
    s3 = adaptive_threshold(blur_s2, strategy, pct=pct_val)
    masks.append(s3)

    # Stage 4: + morph open
    s4 = cv2.morphologyEx(s3, cv2.MORPH_OPEN, k, iterations=params["open_iterations"])
    masks.append(s4)

    # Stage 5: + morph close + hole fill
    s5 = cv2.morphologyEx(s4, cv2.MORPH_CLOSE, k, iterations=params["close_iterations"])
    s5 = (binary_fill_holes(s5 > 0) * 255).astype(np.uint8)
    masks.append(s5)

    # Stage 6: + size filter
    s6  = s5.copy()
    lbl, n = ndi.label(s6 > 0)
    for i in range(1, n + 1):
        pix = lbl == i
        if pix.sum() < params["min_area_px"] or pix.sum() > params["max_area_px"]:
            s6[pix] = 0
    masks.append(s6)

    # Compute metrics at each stage
    for m in masks:
        met = compute_metrics(m, gt_mask)
        ious.append(met["IoU"]); dices.append(met["Dice"])

    # [NEW-3] Bar chart
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))
    fig.suptitle(f"Step-by-Step Performance  —  {img_name}",
                 fontsize=12, fontweight="bold")

    x      = np.arange(len(stages_names))
    width  = 0.38
    clrs_i = ["#90CAF9"] * (len(x)-1) + ["#1565C0"]   # last bar darker
    clrs_d = ["#FFCC80"] * (len(x)-1) + ["#E65100"]

    ax = axes[0]
    bars_i = ax.bar(x - width/2, ious,  width, color=clrs_i, edgecolor="k", lw=0.7, label="IoU")
    bars_d = ax.bar(x + width/2, dices, width, color=clrs_d, edgecolor="k", lw=0.7, label="Dice")
    ax.set_xticks(x); ax.set_xticklabels(stages_names, fontsize=8.5, rotation=15, ha='right')
    ax.set_ylim(0, 1.1); ax.set_ylabel("Score"); ax.set_title("IoU & Dice per Stage")
    ax.legend(fontsize=9)
    
    # annotate bars
    for bar, val in zip(list(bars_i)+list(bars_d), ious+dices):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.02,
                f"{val:.3f}", ha="center", fontsize=7.5, fontweight="bold")

    # delta from baseline
    ax2 = axes[1]
    delta_i = [v - ious[0] for v in ious]
    delta_d = [v - dices[0] for v in dices]
    cols = ["#43A047" if d >= 0 else "#E53935" for d in delta_i]
    ax2.bar(x - width/2, delta_i, width, color=cols,        edgecolor="k", lw=0.7, label="ΔIoU")
    cols_d = ["#FB8C00" if d >= 0 else "#8E24AA" for d in delta_d]
    ax2.bar(x + width/2, delta_d, width, color=cols_d,      edgecolor="k", lw=0.7, label="ΔDice")
    ax2.axhline(0, color="k", lw=0.8)
    ax2.set_xticks(x); ax2.set_xticklabels(stages_names, fontsize=8.5, rotation=15, ha='right')
    ax2.set_ylabel("Δ vs baseline (Stage 0)"); ax2.set_title("Incremental Contribution of Each Step")
    ax2.legend(fontsize=9)

    plt.tight_layout()
    if save_prefix:
        p = f"{save_prefix}_stepwise_iou.png"
        plt.savefig(p, dpi=150, bbox_inches="tight"); print(f"    → {p}")
    plt.show(); plt.close()

    # [NEW-4] Visual grid of intermediate binary masks
    n_cols = len(stages_names)
    fig2, axes2 = plt.subplots(3, n_cols, figsize=(n_cols * 3.2, 9))
    fig2.suptitle(f"Intermediate Masks — {img_name}", fontsize=12, fontweight="bold")

    gt_b = gt_mask > 0
    for j, (name_s, mask_s, iou_s, dice_s) in enumerate(
            zip(stages_names, masks, ious, dices)):
        # Row 0: mask
        axes2[0, j].imshow(mask_s, cmap="gray")
        axes2[0, j].set_title(name_s, fontsize=7.5)

        # Row 1: error map TP/FP/FN
        pb   = mask_s > 0
        err  = np.zeros((*mask_s.shape, 3), np.uint8)
        err[ np.logical_and( pb,  gt_b)] = [0,   200,   0]
        err[ np.logical_and( pb, ~gt_b)] = [220,   0,   0]
        err[ np.logical_and(~pb,  gt_b)] = [0,     0, 220]
        axes2[1, j].imshow(err)
        axes2[1, j].set_title(f"IoU={iou_s:.3f} Dice={dice_s:.3f}", fontsize=7.5)

        # Row 2: overlay on original
        ov = image_rgb.copy().astype(np.float32)
        ov[mask_s > 0] = ov[mask_s > 0]*0.45 + np.array([255,165,0])*0.55
        axes2[2, j].imshow(ov.astype(np.uint8))
        axes2[2, j].set_title("Overlay", fontsize=7.5)

    for ax in axes2.flat:
        ax.axis("off")

    plt.tight_layout()
    if save_prefix:
        p = f"{save_prefix}_stepwise_grid.png"
        plt.savefig(p, dpi=150, bbox_inches="tight"); print(f"    → {p}")
    plt.show(); plt.close()

    return dict(stages=stages_names, ious=ious, dices=dices)

### Final Result Visualiser

In [ ]:
def visualise_result(image_rgb, gt_mask, pred_mask, metrics,
                     title="", save_path=None):
    """6-panel final result figure (enhanced from V1)"""
    fig, axes = plt.subplots(2, 3, figsize=(18, 11))
    fig.suptitle(f"{title} IoU = {metrics['IoU']:.4f}  |  Dice = {metrics['Dice']:.4f}" f"  |  Prec = {metrics['Precision']:.4f}  |  Rec = {metrics['Recall']:.4f}", fontsize=12, fontweight="bold")

    def blend(img, mask, colour, alpha=0.55):
        ov = img.astype(np.float32).copy()
        ov[mask > 0] = ov[mask > 0]*(1-alpha) + np.array(colour)*alpha
        return np.clip(ov, 0, 255).astype(np.uint8)

    axes[0,0].imshow(image_rgb);                axes[0,0].set_title("Original H&E")
    axes[0,1].imshow(gt_mask,   cmap="Greens"); axes[0,1].set_title("Ground Truth")
    axes[0,2].imshow(pred_mask, cmap="Oranges");axes[0,2].set_title("Predicted Mask")
    axes[1,0].imshow(blend(image_rgb, gt_mask,   [0,220,0]));   axes[1,0].set_title("GT Overlay")
    axes[1,1].imshow(blend(image_rgb, pred_mask, [255,140,0])); axes[1,1].set_title("Prediction Overlay")

    # Error map
    g  = gt_mask > 0; p  = pred_mask > 0
    em = np.zeros((*gt_mask.shape, 3), np.uint8)
    em[ np.logical_and( p,  g)] = [0,   210,  0]
    em[ np.logical_and( p, ~g)] = [220,   0,  0]
    em[ np.logical_and(~p,  g)] = [0,     0, 220]
    axes[1,2].imshow(em)
    axes[1,2].legend(
        handles=[mpatches.Patch(color="#00D200", label=f"TP {metrics['TP']:,}"),
                 mpatches.Patch(color="#DC0000", label=f"FP {metrics['FP']:,}"),
                 mpatches.Patch(color="#0000DC", label=f"FN {metrics['FN']:,}")],
        loc="lower right", fontsize=9)
    axes[1,2].set_title("Error Map  (TP / FP / FN)")

    for ax in axes.flat: ax.axis("off")
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        print(f"    → {save_path}")
    plt.show(); plt.close()


def load_image(img_path: Path) -> np.ndarray:
    """Load TIFF/image → uint8 RGB (H, W, 3)"""
    img = tifffile.imread(str(img_path)) if _USE_TIFFFILE else           cv2.cvtColor(cv2.imread(str(img_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
    if img.ndim == 2:           img = np.stack([img]*3, -1)
    elif img.ndim == 3 and img.shape[2] > 3: img = img[:, :, :3]
    if img.dtype != np.uint8:
        img = img.astype(np.float32)
        img = ((img-img.min())/(img.max()-img.min()+1e-8)*255).astype(np.uint8)
    return img


### [NEW-5]  Running-Time Diagnostic: Per-Stage Profiling

`run_timing_diagnostic()` re-executes the same operations used inside `segment_nuclei()`, 
stain extraction (`rgb2hed`), CLAHE, Gaussian blur, adaptive threshold, morphological
open/close, hole filling, connected-component labelling, and the size-filter loop, each
wrapped with `time.perf_counter()`. **`segment_nuclei()` itself is left completely
unmodified**; this is a read-only instrumentation pass that answers the question
*"where does the wall-clock time actually go?"*.

For every image it produces a horizontal bar chart of per-stage time (averaged over several
repeats) and reports the number of connected components found right before size filtering,
this component count is the main driver of the size-filter loop's cost (full discussion in
Section 11).

In [ ]:
def run_timing_diagnostic(image_rgb: np.ndarray,
                          img_name: str = "",
                          params: dict = PARAMS,
                          save_prefix: str = None,
                          n_repeats: int = 3):
    """
    [NEW-5]  Profile wall-clock time of every stage used inside segment_nuclei().

    This duplicates the *operations* of segment_nuclei() purely for timing — it does not
    call or modify segment_nuclei() itself, and its output mask is not used for evaluation

    Returns
    -------
    dict(stages=<labels>, times_ms=<mean ms per stage>, total_ms=<sum>,
         n_components=<connected components before size filtering>)
    """
    stage_labels = [
        "1. Stain extraction\n(rgb2hed)",
        "2. CLAHE",
        "3. Gaussian blur",
        "4. Adaptive threshold",
        "5. Morph open+close",
        "6. Hole fill",
        "7. CC labeling",
        "8. Size-filter loop",
    ]
    times = {lbl: [] for lbl in stage_labels}

    strategy = THRESHOLD_MAP.get(img_name, "otsu")
    pct_val  = PERCENTILE_MAP.get(img_name, 60)
    ks = params["morph_kernel_size"]
    k  = np.ones((ks, ks), np.uint8)

    n_cc = 0
    for _ in range(n_repeats):
        t0 = time.perf_counter()
        H_u8, _ = get_h_channel(image_rgb)
        times[stage_labels[0]].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        if params.get("use_clahe", True):
            clahe = cv2.createCLAHE(clipLimit=params["clahe_clip_limit"],
                                    tileGridSize=params["clahe_tile_size"])
            H_cl  = clahe.apply(H_u8)
        else:
            H_cl = H_u8
        times[stage_labels[1]].append(time.perf_counter() - t0)

        t0   = time.perf_counter()
        blur = cv2.GaussianBlur(H_cl, params["gaussian_ksize"], 0)
        times[stage_labels[2]].append(time.perf_counter() - t0)

        t0     = time.perf_counter()
        binary = adaptive_threshold(blur, strategy, pct=pct_val)
        times[stage_labels[3]].append(time.perf_counter() - t0)

        t0 = time.perf_counter()
        binary_m = cv2.morphologyEx(binary,   cv2.MORPH_OPEN,  k, iterations=params["open_iterations"])
        binary_m = cv2.morphologyEx(binary_m, cv2.MORPH_CLOSE, k, iterations=params["close_iterations"])
        times[stage_labels[4]].append(time.perf_counter() - t0)

        t0       = time.perf_counter()
        binary_f = (binary_fill_holes(binary_m > 0) * 255).astype(np.uint8)
        times[stage_labels[5]].append(time.perf_counter() - t0)

        t0            = time.perf_counter()
        lbl_arr, n_cc = ndi.label(binary_f > 0)
        times[stage_labels[6]].append(time.perf_counter() - t0)

        t0  = time.perf_counter()
        out = binary_f.copy()
        for i in range(1, n_cc + 1):
            pix = lbl_arr == i
            if pix.sum() < params["min_area_px"] or pix.sum() > params["max_area_px"]:
                out[pix] = 0
        times[stage_labels[7]].append(time.perf_counter() - t0)

    vals_ms = [np.mean(times[lbl]) * 1000 for lbl in stage_labels]
    total   = sum(vals_ms)

    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(stage_labels, vals_ms, color="#4C72B0", edgecolor="k")
    ax.set_xlabel(f"Time (ms) — averaged over {n_repeats} run(s)")
    ax.set_title(f"Per-Stage Running-Time Breakdown — {img_name}\n"
                 f"Total ≈ {total:.1f} ms  |  Connected components before size filter: {n_cc}")
    for bar, v in zip(bars, vals_ms):
        ax.text(bar.get_width() + max(vals_ms) * 0.01, bar.get_y() + bar.get_height()/2,
                f"{v:.1f} ms ({v/total*100:.0f}%)", va="center", fontsize=8.5)
    ax.invert_yaxis()
    plt.tight_layout()
    if save_prefix:
        p = f"{save_prefix}_timing_breakdown.png"
        plt.savefig(p, dpi=150, bbox_inches="tight"); print(f"    → {p}")
    plt.show(); plt.close()

    return dict(stages=stage_labels, times_ms=vals_ms, total_ms=total, n_components=n_cc)

## 9 · Main Processing Loop

In [ ]:
#  Set to True to generate full diagnostic plots (slower)
RUN_DIAGNOSTICS = True

all_results    = []
step_results   = {}   # stores stepwise IoU/Dice data per image
timing_results = {}   # [NEW-5] stores per-stage timing data per image
DEVICE_TAG     = "GPU" if GPU_AVAILABLE else "CPU"
print(f"Device : {DEVICE_TAG}  |  Stain mode : {STAIN_MODE}  |  Watershed : {USE_WATERSHED}\n")

for name in IMAGE_NAMES:
    print("=" * 68)
    print(f"  {name}")
    print("=" * 68)

    img_path = TISSUE_DIR / f"{name}.tif"
    xml_path = ANNOTATION_DIR / f"{name}.xml"
    pfx      = str(OUTPUT_DIR / name)

    image   = load_image(img_path)
    print(f"    Shape: {image.shape}  dtype: {image.dtype}")
    gt_mask = parse_xml_to_mask(xml_path, image.shape)

    # [NEW-1] RGB Histogram
    H_u8_raw, _ = get_h_channel(image)
    plot_rgb_histogram(image, H_u8_raw, title=name,
                       save_path=f"{pfx}_histogram.png")

    # [NEW-2] CLAHE Diagnostic and [MOD-5] now run for every image
    plot_clahe_diagnostic(H_u8_raw, PARAMS, title=name,
                          save_path=f"{pfx}_clahe_diagnostic.png")

    # [NEW-3] [NEW-4] Step-by-step diagnostic
    if RUN_DIAGNOSTICS:
        step_res = run_stepwise_diagnostic(
            image, gt_mask, img_name=name,
            params=PARAMS, save_prefix=pfx)
        step_results[name] = step_res

    # Main segmentation (timed)
    t0        = time.perf_counter()
    pred_mask = segment_nuclei(image, img_name=name,
                               params=PARAMS, use_gpu=GPU_AVAILABLE)
    elapsed   = time.perf_counter() - t0

    # Metrics
    m = compute_metrics(pred_mask, gt_mask)
    print(f"    IoU  (Jaccard)  : {m['IoU']:.4f}")
    print(f"    Dice Similarity : {m['Dice']:.4f}")
    print(f"    Precision       : {m['Precision']:.4f}")
    print(f"    Recall          : {m['Recall']:.4f}")
    print(f"    Running Time    : {elapsed:.4f} s  [{DEVICE_TAG}]")

    # Final visualisation
    visualise_result(image, gt_mask, pred_mask, m,
                     title=name,
                     save_path=f"{pfx}_result.png")

    all_results.append({
        "Image"       : name,
        "IoU"         : round(m["IoU"],       4),
        "Dice"        : round(m["Dice"],      4),
        "Precision"   : round(m["Precision"], 4),
        "Recall"      : round(m["Recall"],    4),
        "Running Time": round(elapsed,        4),
        "Threshold"   : THRESHOLD_MAP.get(name, "otsu"),
        "Device"      : DEVICE_TAG,
    })

    # [NEW-5] Per-stage running-time diagnostic
    timing_res = run_timing_diagnostic(image, img_name=name,
                                       params=PARAMS, save_prefix=pfx)
    timing_results[name] = timing_res

    print()

## 10 · Results Summary

In [ ]:
df = pd.DataFrame(all_results)

print("\n" + "=" * 74)
print("  FINAL RESULTS SUMMARY  (Version 2)")
print("=" * 74)
print(df[["Image", "IoU", "Dice", "Precision", "Recall",
          "Running Time", "Threshold", "Device"]].to_string(index=False))
print("=" * 74)
print(f"  Mean IoU   : {df['IoU'].mean():.4f}  (± {df['IoU'].std():.4f})")
print(f"  Mean Dice  : {df['Dice'].mean():.4f}  (± {df['Dice'].std():.4f})")
print(f"  Mean Time  : {df['Running Time'].mean():.4f} s")
print("=" * 74)

csv_path = OUTPUT_DIR / "results_v2.csv"
df.to_csv(csv_path, index=False)
print(f"\n  → CSV saved : {csv_path}")

print("\n" + "═"*55)
print("  >>> VALUES FOR GOOGLE FORM LEADERBOARD <<<")
print("═"*55)
for row in all_results:
    print(f"  {row['Image']}")
    print(f"    IoU          : {row['IoU']}")
    print(f"    Dice         : {row['Dice']}")
    print(f"    Running Time : {row['Running Time']} s\n")


In [ ]:
# Performance bar chart
short   = [n.split("-")[1] + "\n" + n.split("-")[2][:5] for n in df["Image"]]
colours = ["#2196F3", "#4CAF50", "#FF9800", "#E91E63"]

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.suptitle("Nucleus Segmentation V2 — Performance per Image",
             fontsize=13, fontweight="bold")

for ax, col, ylabel in zip(
    axes,
    ["IoU", "Dice", "Running Time"],
    ["IoU (Jaccard)", "Dice Similarity Coefficient", "Running Time (s)"]):
    bars = ax.bar(short, df[col], color=colours, edgecolor="k", lw=0.7)
    ax.set_title(ylabel, fontsize=11)
    ax.set_ylim(0, max(df[col]) * 1.28)
    ax.axhline(df[col].mean(), color="red", ls="--", lw=1.5,
               label=f"Mean={df[col].mean():.3f}")
    ax.legend(fontsize=9)
    for bar, val in zip(bars, df[col]):
        ax.text(bar.get_x()+bar.get_width()/2,
                bar.get_height()+max(df[col])*0.015,
                f"{val:.4f}", ha="center", fontsize=8.5, fontweight="bold")

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "performance_v2.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# [NEW-3]  Cross-image step-IoU comparison
if step_results:
    n_stages = len(list(step_results.values())[0]["stages"])
    stage_labels = list(step_results.values())[0]["stages"]
    x     = np.arange(n_stages)
    width = 0.20
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    fig.suptitle("Step-by-Step Performance — All Images", fontsize=13, fontweight="bold")

    pal = ["#2196F3","#4CAF50","#FF9800","#E91E63"]
    for idx, (name, sr) in enumerate(step_results.items()):
        short_n = name.split("-")[1]
        off     = (idx - 1.5) * width
        axes[0].bar(x + off, sr["ious"],  width, color=pal[idx],
                    alpha=0.85, edgecolor="k", lw=0.6, label=short_n)
        axes[1].bar(x + off, sr["dices"], width, color=pal[idx],
                    alpha=0.85, edgecolor="k", lw=0.6, label=short_n)

    for ax, metric in zip(axes, ["IoU per Stage", "Dice per Stage"]):
        ax.set_xticks(x); ax.set_xticklabels(stage_labels, fontsize=8)
        ax.set_ylim(0, 1.05); ax.set_ylabel("Score")
        ax.set_title(metric); ax.legend(fontsize=9, title="Image")
        ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "stepwise_all_images.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("→ Saved: stepwise_all_images.png")


### [NEW-5]  Cross-Image Running-Time Comparison

Same per-stage timings as above, plotted side-by-side for all four images. This makes it
immediately clear that **Stage 8 (size-filter loop) consistently dominates total runtime**,
and that its cost scales with the number of connected components produced after
thresholding/morphology, *not* with anything GPU acceleration would help with (the
GPU/CuPy code path inside `segment_nuclei()` is only reached when `USE_WATERSHED=True`,
which is `False` by default).

In [ ]:
# [NEW-5]  Cross-image running-time comparison
if timing_results:
    stage_labels_t = list(timing_results.values())[0]["stages"]
    xt     = np.arange(len(stage_labels_t))
    widtht = 0.20
    fig, ax = plt.subplots(figsize=(13, 6))
    fig.suptitle("Per-Stage Running Time — All Images", fontsize=13, fontweight="bold")

    pal = ["#2196F3","#4CAF50","#FF9800","#E91E63"]
    for idx, (name, tr) in enumerate(timing_results.items()):
        short_n = name.split("-")[1]
        off     = (idx - 1.5) * widtht
        ax.bar(xt + off, tr["times_ms"], widtht, color=pal[idx],
               alpha=0.85, edgecolor="k", lw=0.6, label=short_n)

    ax.set_xticks(xt); ax.set_xticklabels(stage_labels_t, fontsize=8.5, rotation=15, ha="right")
    ax.set_ylabel("Time (ms)"); ax.set_title("Per-Stage Time (ms, mean of 3 runs)")
    ax.legend(fontsize=9, title="Image")
    ax.grid(axis="y", alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "timing_breakdown_all_images.png", dpi=150, bbox_inches="tight")
    plt.show()
    print("→ Saved: timing_breakdown_all_images.png")

    print("\nShare of total time spent in Stage 8 (size-filter loop):")
    for name, tr in timing_results.items():
        share = tr["times_ms"][-1] / tr["total_ms"] * 100
        print(f"  {name:28s}: {share:5.1f} %   ({tr['n_components']} components before filtering)")

## 11 · Detailed Modification Notes to see (carry over from V2 with minor changes regarding the new V2.1)

### [MOD-1]  Stain Extraction `skimage.rgb2hed` + Macenko
- **V1**: manually built the 3×3 Ruifrok-Johnston matrix and inverted it in NumPy.
- **V2**: `skimage.color.rgb2hed` applies the same matrix but via a well-tested,
  optimised path with proper float handling.
- **Macenko** (optional, `STAIN_MODE = "macenko"`): runs SVD on the optical-density
  cloud of tissue pixels from *this specific image*, finding the actual H and E
  directions. Useful when staining differs significantly from the standard.

### [MOD-2]  Adaptive Threshold
- **V1**: `Otsu_value × otsu_factor` (fixed global multiplier).
- **V2**: per-image strategy dict (`THRESHOLD_MAP`):
  - `"otsu"`: standard Otsu; works when nucleus vs background form a clear bimodal histogram.
  - `"percentile"`: 75th-percentile of the blurred H channel; used for `TCGA-AY-A8YK`
    where nuclei are the majority class and Otsu therefore picks too high a threshold,
    leaving most nuclei un-detected.
  - `"auto"`: computes the Otsu foreground fraction; if > 60 % of pixels would be
    classified as foreground, switches to percentile automatically.

### [MOD-3]  Watershed, now optional
- **V1**: always applied watershed with distance-transform markers.
- **V2**: `USE_WATERSHED = False` by default. Watershed can cause *over-segmentation*
  when the distance-transform produces many closely-spaced local maxima
  (common in dense tissue), creating extra boundaries that lower IoU relative to GT.
  The simpler morphological path achieves comparable or better IoU for this dataset.
  Set `USE_WATERSHED = True` to re-enable if you observe under-segmented touching nuclei.

### [MOD-4]  Morphological Operations
- **V1**: elliptical structuring element, open radius=2, close radius=3, single-pass.
- **V2**: square 3×3 kernel with `iterations=2` for both open and close.
  This matches the approach from the reference pipeline and provides slightly stronger
  gap-filling / noise removal without introducing large structural distortions.

### [NEW-1] RGB + H-Channel Histogram
Plots the per-channel RGB distribution of the original image alongside the H-channel
histogram. Red vertical lines show the Otsu and 75th-percentile candidates, making it
immediately visible whether the histogram is bimodal (Otsu safe) or unimodal (percentile better).

### [NEW-2] CLAHE Diagnostic
Shows the H-channel image and its histogram before and after CLAHE, along with the CDF.
A CDF that is closer to a straight diagonal line after CLAHE means pixel values are more
uniformly distributed, better dynamic range usage, which helps the subsequent threshold
find a cleaner boundary.

### [NEW-3] Step-by-Step IoU/Dice Bar Chart
For each image, records IoU and Dice at 7 pipeline stages and plots:
- Absolute values (left panel), absolute score at each stage
- Incremental delta (right panel), gain/loss relative to the Stage-0 baseline

### [NEW-4] Step Visual Grid
Three-row grid per image:  
Row 0: binary mask at each stage  
Row 1: error map (TP=green, FP=red, FN=blue) at each stage  
Row 2: prediction overlay on original H&E  
Makes it visually clear when a step improves or degrades the segmentation.


### [MOD-5]  CLAHE Diagnostic, Now Generated for Every Image
- **V2**: `plot_clahe_diagnostic()` was only called for `IMAGE_NAMES[0]`, "for brevity".
- **V2.1**: the `if name == IMAGE_NAMES[0]` guard in the main loop is removed, so the
  before/after CLAHE images, histograms, and CDF curves are produced for **all four images**.
  This matters because each image has a different staining intensity, so the contrast gain
  from CLAHE, and therefore its effect on the downstream threshold, is not the same across
  images; inspecting only `TCGA-AR-A1AS` hid that variability.

### [NEW-5]  Running-Time Diagnostic, Where Does the Time Actually Go?
`run_timing_diagnostic()` times each stage of the pipeline separately (averaged over 3 runs)
without altering `segment_nuclei()`. Two consistent findings hold across all four images:

1. **Stage 8 (size-filter loop) dominates the total time** — roughly half to nearly 90% of it,
   depending on the image. The loop is:
   ```python
   for i in range(1, n + 1):
       pix = lbl == i
       if pix.sum() < min_area_px or pix.sum() > max_area_px:
           binary[pix] = 0
   ```
   For an `n`-component label map of an `H×W` image this is **O(n·H·W)**: every component
   re-scans the *entire* image. On a 1000×1000 image with a few hundred components, typical
   after percentile thresholding, since `close_iterations = 0` leaves many small specks
   un-merged, this single loop alone can cost several hundred milliseconds to ~1 s.

2. **The GPU path is never executed.** Inside `segment_nuclei()`, `cp` / `cpnd` are only
   referenced inside `if use_ws:` (the watershed branch). Since `USE_WATERSHED = False`,
   that branch never runs, so enabling CuPy/GPU has **zero effect** on the measured runtime,
   every stage that *does* run (`rgb2hed`, CLAHE, blur, threshold, morphology, hole-filling,
   labeling, size filter) is plain NumPy / OpenCV / SciPy on the CPU.

Together, these two facts explain why total running time sits in the 1–4 s range per image
even with a GPU available. A reference implementation reporting ~0.1 s very likely produces
far fewer connected components and/or replaces the Python-level area-filter loop with a
vectorised count (e.g. `np.bincount` on the label array + a boolean lookup table), which is
`O(H·W + n)` instead of `O(n·H·W)`. This diagnostic does **not** change `segment_nuclei()`,
it only documents the measured behaviour of the existing, validated pipeline.
